# FirePath AI on Google Colab

Run top to bottom. Use **Runtime → Change runtime type → T4 GPU** for training.

Part B (simulation) works immediately with placeholder numbers. Part A trains the detector and measures its real errors, which replace the placeholder.

## 0. Get the project
Upload `firepath.zip` with the folder icon on the left, then run:

In [ ]:
!unzip -q -o firepath.zip -d /content
%cd /content/firepath
!pip install -q -r requirements.txt

## 1. Quick test of the simulation (placeholder numbers, NOT results)

In [ ]:
!python -m firepath.experiments --scenarios 60 --skip-ablation --out results_demo
from IPython.display import Image, display
display(Image('results_demo/tradeoff.png'))

In [ ]:
!python -m firepath.visualize --find-interesting --out results_demo/route_example.png
display(Image('results_demo/route_example.png'))

## 2. Get D-Fire
Download D-Fire from the dataset's GitHub page (gaiasd/DFireDataset). The easiest way is to put the zip in your Google Drive and copy it here. Edit the path below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o '/content/drive/MyDrive/D-Fire.zip' -d /content/D-Fire
!ls /content/D-Fire

Check which class id is fire and which is smoke. The output tells you what to pass as `--names`.

In [ ]:
!python detection/check_dataset.py --root /content/D-Fire

## 3. Fine-tune YOLO (a few hours)
Change `--names` if step 2 said so. Start with fewer epochs (e.g. 10) to test, then run 50.

In [ ]:
!python detection/train_yolo.py --root /content/D-Fire --names smoke fire --epochs 50 --model yolov8n.pt

Copy the weights to Drive so a disconnect doesn't lose them:

In [ ]:
!mkdir -p /content/drive/MyDrive/firepath && cp runs/firepath/train/weights/best.pt /content/drive/MyDrive/firepath/

## 4. Measure the detector's real errors on videos
Put evaluation videos (NOT D-Fire training images) in Drive and list them in `videos.csv` (see `configs/videos_template.csv`). Labels: `fire`, `smoke`, `none` (normal scene), `confuser` (fire-like scene such as a sunset or red light). A separate `calib.csv` for calibration is recommended.

In [ ]:
!python detection/measure_detector.py --weights runs/firepath/train/weights/best.pt \
    --videos /content/drive/MyDrive/firepath/videos.csv \
    --calib-videos /content/drive/MyDrive/firepath/calib.csv \
    --out configs/detector_errors_measured.json
!cp configs/detector_errors_measured.* /content/drive/MyDrive/firepath/

## 5. Choose λ on validation scenarios, then run the real experiments

In [ ]:
!python -m firepath.experiments --errors configs/detector_errors_measured.json --seed0 100000 --scenarios 200 --skip-ablation --out results_val
display(Image('results_val/tradeoff.png'))

In [ ]:
!python -m firepath.experiments --errors configs/detector_errors_measured.json --seed0 0 --scenarios 300 --out results_test
print(open('results_test/summary.md').read())
print(open('results_test/error_types.md').read())
display(Image('results_test/tradeoff.png'))
!cp -r results_test /content/drive/MyDrive/firepath/